# PornWorks Web Generator — Colab Prototype

A small browser-based interface for the PornWorks Real Porn V03 SDXL checkpoint. Run this notebook in Google Colab, then open the temporary Gradio link it prints.

**Before starting:** In Colab, choose **Runtime → Change runtime type → T4 GPU** if available. The public Gradio link is temporary and anyone who has it may be able to access the running demo. Use only fictional consenting adults; never create or upload intimate imagery of a real person without their explicit consent.

In [ ]:
!pip -q install -U diffusers transformers accelerate safetensors gradio

In [ ]:
import torch
import gradio as gr
from diffusers import DiffusionPipeline

MODEL_ID = "John6666/pornworks-real-porn-v03-sdxl"
has_cuda = torch.cuda.is_available()
dtype = torch.float16 if has_cuda else torch.float32

print("GPU available:", has_cuda)
if has_cuda:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Generation will be very slow; switch Colab to a GPU runtime if possible.")

pipe = DiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    use_safetensors=True,
)

if has_cuda:
    pipe.enable_model_cpu_offload()
    pipe.enable_attention_slicing()
else:
    pipe.to("cpu")

pipe.vae.enable_slicing()
pipe.vae.enable_tiling()
print("Model loaded.")

In [ ]:
import random

def generate_image(prompt, negative_prompt, width, height, steps, guidance, seed):
    prompt = (prompt or "").strip()
    if not prompt:
        raise gr.Error("Enter a prompt first.")
    if width % 8 or height % 8:
        raise gr.Error("Width and height must be divisible by 8.")
    actual_seed = int(seed)
    if actual_seed < 0:
        actual_seed = random.randint(0, 2**31 - 1)
    generator_device = "cuda" if has_cuda else "cpu"
    generator = torch.Generator(device=generator_device).manual_seed(actual_seed)
    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt or None,
        width=int(width),
        height=int(height),
        num_inference_steps=int(steps),
        guidance_scale=float(guidance),
        generator=generator,
    )
    return result.images[0], f"Seed: {actual_seed}"

custom_css = """
body { background: #101014; }
.gradio-container { max-width: 1100px !important; }
#hero { text-align: center; padding: 18px 0 8px; }
#hero h1 { letter-spacing: -1px; }
"""

with gr.Blocks(theme=gr.themes.Soft(primary_hue="violet", neutral_hue="slate"), css=custom_css, title="PornWorks Image Lab") as demo:
    gr.Markdown("# PornWorks Image Lab\nCreate photorealistic images with the Real Porn V03 SDXL model. Use fictional consenting adults only.", elem_id="hero")
    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(label="Prompt", placeholder="Describe the fictional adult characters, scene, lighting, camera, and style…", lines=5)
            negative = gr.Textbox(label="Negative prompt", value="blurry, low quality, distorted anatomy, extra fingers, watermark, text", lines=2)
            with gr.Accordion("Generation settings", open=True):
                width = gr.Slider(512, 1024, value=768, step=64, label="Width")
                height = gr.Slider(512, 1024, value=768, step=64, label="Height")
                steps = gr.Slider(10, 50, value=28, step=1, label="Steps")
                guidance = gr.Slider(1, 10, value=5.0, step=0.5, label="Guidance scale")
                seed = gr.Number(value=-1, precision=0, label="Seed (-1 = random)")
            run = gr.Button("Generate image", variant="primary")
        with gr.Column(scale=1):
            output = gr.Image(label="Generated image", type="pil")
            seed_output = gr.Textbox(label="Generation info", interactive=False)
    run.click(generate_image, inputs=[prompt, negative, width, height, steps, guidance, seed], outputs=[output, seed_output])
    gr.Markdown("**Privacy note:** This prototype does not intentionally save generated images to a database, but prompts and outputs may be processed by the hosted runtime. Do not enter personal or confidential information.")

demo.queue(max_size=2).launch(share=True, debug=True)

## How to use
1. Run each cell from top to bottom.
2. Wait for the model to finish loading.
3. Open the `gradio.live` link printed by the last cell.
4. Keep the Colab tab and runtime running while using the temporary site.

The share link expires when the runtime stops. Do not share it publicly unless you are comfortable with other people accessing the demo.